In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle


def ReadFile(name):
    d = np.loadtxt(name)
    return d[0, 1:], d[1:, 0], d[1:, 1:]     # X, Y, M[j, i]


X, Y, Phi = ReadFile('PhiMatrix.dat')
_, _, dx  = ReadFile('dxPhiMatrix.dat')
_, _, dy  = ReadFile('dyPhiMatrix.dat')

Ex, Ey = -dx, -dy                             # E = -grad(Phi)
X, Y = 1000*X, 1000*Y                         # convert to mm
XX, YY = np.meshgrid(X, Y)
theta = np.arctan2(YY, XX)
# Deep-metal nodes are exactly 0 in both components: hide them
mask = (dx == 0) & (dy == 0)
Phi = np.ma.array(Phi, mask=mask) 
Ex = np.ma.array(Ex, mask=mask)
Ey = np.ma.array(Ey, mask=mask)
Dcentral_Phi = np.cos(theta)*dx + np.sin(theta)*dy

p = np.pad(mask,1,constant_values=False)
near = np.zeros_like(mask, dtype=bool)
near = (p[2:, 1:-1] | p[:-2, 1:-1] | p[1:-1, 2:] | p[1:-1, :-2] |    
        p[2:, 2:]   | p[2:, :-2]   | p[:-2, 2:]  | p[:-2, :-2]) & ~mask
Phi_ring = np.ma.array(Phi, mask=~near)
Dcentral_Phi_borders = np.ma.array(Dcentral_Phi, mask=~near)



origin = np.argmin(XX**2 + YY**2,axis=None)  
origin = np.unravel_index(origin, XX.shape) 
Phi_origin = Phi[origin]

In [23]:
rr = np.hypot(XX, YY)
print(rr[near].mean(), rr[near].std())  
print(Phi_origin, dx[origin], dy[origin], Dcentral_Phi[origin])
with open('DataOutput.txt', 'w') as f:
    f.write(f'Phi(0)={Phi_origin}\n')
    f.write(f'dx(0)={dx[origin]}\n')
    f.write(f'dy(0)={dy[origin]}\n')
    f.write(f'Dcentral_Phi(0)={Dcentral_Phi[origin]}\n')


19.980161206245455 0.02140489625381749
-0.62272581 -0.0099607215 -0.40838788 0.2958171330205587
